# Qiskit Workshop — Day 0: Setup

**Goal:** get every laptop (Windows / macOS / Linux) to the *exact same* working
Qiskit environment so Day 1 runs without "it works on my machine" problems.

By the end of this notebook you will have:

- A clean, isolated Python environment (conda **or** venv)
- Qiskit + Aer simulator + IBM Runtime installed and version-checked
- **Graphviz** working, so you can *visualize transpiler pass managers*
- A Bell state run on a **local simulator**
- (Optional) an **IBM Quantum** account connected for **real hardware**

> Tested against: `qiskit 2.4.x`, `qiskit-aer 0.17.x`, `qiskit-ibm-runtime 0.47.x`, Python 3.10–3.13.

---

### Note 
For the ease of use, we recommend you use [qBRAID]() however feel free to run the notebooks from this workshop locally, you will find the steps for the setup below. [Here](https://quantum.cloud.ibm.com/docs/en/guides/online-lab-environments) are a few other approaches for the initial setup. 

---

**Before Step 1: if you do **not** already have conda or a Python with `venv`, install one first**

Step 1 assumes you already have **either** `conda` **or** a Python installation that includes the built-in `venv` module.

- **Want the easiest path, especially on Windows?** Install **Miniconda**: https://docs.conda.io/en/latest/miniconda.html
- **Prefer plain Python + `venv`?** Install Python 3.10+ from https://www.python.org/downloads/
- **Windows:** during Python install, make sure **Add Python to PATH** is enabled.
- **macOS:** after installing Python from python.org, reopen Terminal so `python3` is available.
- **Linux:** if `python3 -m venv` fails, install the OS package for venv first (for example Ubuntu/Debian: `sudo apt-get install python3-venv`).

### If you're using qBRAID, do the following commands in a temrinal

In the terminal run the following commands
```bash
curl -Ls https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -o ~/miniconda.sh
bash ~/miniconda.sh -b -p ~/miniconda
source ~/miniconda/etc/profile.d/conda.sh
```

Quick checks in a terminal:

```bash
conda --version      # works if conda/miniconda is installed
python3 --version    # macOS/Linux
py --version         # Windows
python3 -m venv --help
```

If one of those setup routes works, continue to Step 1 below.

## 1. Create an isolated environment (do this in a terminal, *before* opening this notebook)

Never install Qiskit into your system Python. Pick **one** of the two options below.
Both give an identical result; conda is friendlier on Windows.

### Option A — conda / miniconda  *(recommended, esp. Windows)*

Same three commands on **all OSes**:

```bash
conda create -n qiskit-ws python=3.12 -y
conda activate qiskit-ws
conda install -c conda-forge graphviz python-graphviz pydot -y   # installs the Graphviz BINARY too
```

Then install the Python packages (Section 2).

### Option B — built-in `venv` (no conda)

**macOS / Linux**
```bash
python3 -m venv qiskit-ws
source qiskit-ws/bin/activate
```

**Windows (PowerShell)**
```powershell
py -m venv qiskit-ws
qiskit-ws\Scripts\Activate.ps1
```

> With venv you must install the **Graphviz system binary** separately (Section 4) — `pip` only
> installs the Python wrapper, not the `dot` engine that actually draws the graphs.

### Register the env with Jupyter, then launch
```bash
pip install jupyterlab ipykernel
python -m ipykernel install --user --name qiskit-ws --display-name "Python (qiskit-ws)"
```

After launching, re-open this notebook and pick the **"Python (qiskit-ws)"** kernel (top-right).
Run the cell below to confirm you're in the right place — it should point inside `qiskit-ws`, **not** your system Python.

In [ ]:
import sys, platform
print("Python    :", sys.version.split()[0])
print("Executable:", sys.executable)   # should contain 'qiskit-ws'
print("OS        :", platform.system(), platform.release())

## 2. Install the Qiskit stack

`%pip` installs into **this notebook's kernel**, so there's no ambiguity about which
environment gets the packages. Run it once; restart the kernel afterwards if prompted.

| Package | Why |
|---|---|
| `qiskit` | core SDK (circuits, transpiler, primitives) |
| `qiskit-aer` | fast local simulator (statevector / noisy) |
| `qiskit-ibm-runtime` | run on **real IBM hardware** + fake backends |
| `matplotlib`, `pylatexenc` | circuit & histogram drawing |
| `pydot` | bridge Qiskit ↔ Graphviz for pass-manager diagrams |

In [ ]:
%pip install -q \
    "qiskit==2.4.2" \
    "qiskit-aer==0.17.2" \
    "qiskit-ibm-runtime==0.47.0" \
    "matplotlib" "pylatexenc" "pydot" "sympy"
print("\nDone. If pip asked you to restart the kernel, do it now (Kernel > Restart).")

## 3. Verify the install

In [ ]:
import qiskit, qiskit_aer, qiskit_ibm_runtime
import matplotlib, pydot
print("qiskit              :", qiskit.__version__)
print("qiskit-aer          :", qiskit_aer.__version__)
print("qiskit-ibm-runtime  :", qiskit_ibm_runtime.__version__)
print("matplotlib          :", matplotlib.__version__)
print("pydot               :", pydot.__version__)
print("\nAll core imports OK")

## 4. Check Graphviz (the part that trips people up)

Pass-manager diagrams need **two** things:
1. the Python `pydot` package (installed above), and
2. the **Graphviz system binary** (`dot`) on your PATH.

The cell below checks for `dot`. If it's missing, install it for your OS:

| OS | Install the binary |
|---|---|
| conda (any OS) | `conda install -c conda-forge graphviz` |
| macOS (Homebrew) | `brew install graphviz` |
| Ubuntu / Debian | `sudo apt-get install graphviz` |
| Windows (no conda) | `choco install graphviz`  *(or download from graphviz.org and add `bin` to PATH)* |

Restart the kernel after installing the binary so the new PATH is picked up.

In [ ]:
# uncomment the following code if running on qBRAID

import os, sys
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]

import shutil
dot = shutil.which("dot")
if dot:
    print("Graphviz binary found:", dot)
else:
    print("Graphviz binary NOT found")
    print("Pass-manager diagrams won't render. Install it (see table above), then restart the kernel.")

## 5. First run — a Bell state on the local Aer simulator

No account or internet needed. This is the workflow you'll use 90% of the time while learning:

**build circuit → transpile (ISA) → run with a primitive → read results.**

In [ ]:
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as AerSampler

# 1. Build: (|00> + |11>)/sqrt(2)
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# 2. Transpile to the simulator's instruction set (ISA)
sim = AerSimulator()
pm = generate_preset_pass_manager(backend=sim, optimization_level=1)
isa_qc = pm.run(qc)

# 3. Run with the Sampler primitive
sampler = AerSampler()
result = sampler.run([isa_qc], shots=2048).result()

# 4. Read counts (register name from measure_all() is 'meas')
counts = result[0].data.meas.get_counts()
print(counts)

A correct Bell state gives roughly 50/50 over `00` and `11`, with no `01`/`10`.

In [ ]:
from qiskit.visualization import plot_histogram
plot_histogram(counts)

## 6. Visualize transpiler pass managers with Graphviz

In [ ]:
# Draw the staged pipeline. Returns a PIL image when Graphviz is available.
pm_o3 = generate_preset_pass_manager(backend=sim, optimization_level=3)
pm_o3.draw()

In [ ]:
# List the passes inside one stage (the 'optimization' stage at level 3).
# Wrapped defensively because the internal task layout varies across Qiskit versions.
stage = pm_o3.optimization
try:
    tasks = stage.to_flow_controller().tasks
    for t in tasks:
        print(type(t).__name__)
except Exception:
    # Fallback that works on any version: the textual repr lists the passes.
    print(stage)


# Expected output
# ====================
# Size
# Depth
# MinimumPoint
# DoWhileController

Compare optimization levels to *see* how the pipeline changes. Higher levels add
more optimization passes.

In [ ]:
pm_o0 = generate_preset_pass_manager(backend=sim, optimization_level=0)
pm_o0.draw()   # noticeably leaner than the level-3 pipeline above

## 7. Realistic hardware (offline): fake backends

Fake backends carry a **real device's coupling map, basis gates, and noise model** — so you
get hardware-like behavior (and hardware-like transpilation) with no account and no queue.
Great for the whole workshop.

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

fake = FakeManilaV2()
print("Fake backend :", fake.name)
print("Qubits       :", fake.num_qubits)
print("Basis gates  :", fake.operation_names)

# Transpile the SAME Bell circuit for this device — now routing/layout actually matter
pm_hw = generate_preset_pass_manager(backend=fake, optimization_level=1)
isa_hw = pm_hw.run(qc)
print("\nDepth on fake hardware:", isa_hw.depth())

Draw the device-aware pass manager — notice layout/routing stages now do real work:

In [ ]:
pm_hw.draw()

In [ ]:
# Run on the noisy fake backend with the Runtime Sampler (local execution)
from qiskit_ibm_runtime import SamplerV2 as RuntimeSampler

sampler_hw = RuntimeSampler(mode=fake)
res_hw = sampler_hw.run([isa_hw], shots=2048).result()
print(res_hw[0].data.meas.get_counts())   # noise -> a few stray 01/10 counts

## 8. Connect to real IBM Quantum hardware


**One-time setup:**
1. Create a free account at **quantum.cloud.ibm.com**.
2. From the dashboard, copy your **API key**
3. Go to instances from the menu on the top-left where you will find your **CRN**.
4. Run the save cell **once** (it writes `~/.qiskit/qiskit-ibm.json`, in plain text — only on a trusted machine).

In [ ]:
# RUN ONCE, then comment it out. Replace the placeholders.
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    token="",
    instance="",
    set_as_default=True,
    overwrite=True,
)
print("Account saved.")
print("Edit, uncomment, and run this cell only if you have IBM Cloud credentials.")

Once saved, every future session is a one-liner. `channel` defaults to
`ibm_quantum_platform`, so you don't pass it.

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

service = QiskitRuntimeService()                 # uses your saved default account
backend = service.least_busy(operational=True, simulator=False)
print("Least busy real backend:", backend.name, "|", backend.num_qubits, "qubits")

# Transpile FOR that backend, then run with the Sampler:
pm_real = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_real = pm_real.run(qc)
sampler = SamplerV2(mode=backend)
job = sampler.run([isa_real], shots=100)
print("Job ID:", job.job_id())   # results arrive after the queue
print("Uncomment after saving your account.")